# Book Genre Pipeline

This notebook is the centralized place to run the project end to end.
It uses the local Task1 manifests, the extracted `title30cat` image corpus, and the reusable `bookgenre` modules under `src/`.

Suggested workflow:
1. Run setup and config cells.
2. Build datasets and loaders.
3. Train for longer epochs on GPU.
4. Run prediction export.
5. Optionally run clustering on learned embeddings.


In [8]:
# install pre-requisites
!pip install tqdm ipywidgets


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
from pathlib import Path
import json
import sys

import torch
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Project root:", PROJECT_ROOT)
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Project root: C:\Windows\System32
Torch version: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


In [10]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\reddy\OneDrive\Documents\Visual Code\Book Genre Classification\Multimodal-Book-Genre-Classification-Clustering")
sys.path.insert(0, str(PROJECT_ROOT / "src"))


In [11]:
from bookgenre.checkpoints import save_checkpoint_bundle
from bookgenre.clustering import cluster_embeddings, find_cluster_prototypes, sample_silhouette_score
from bookgenre.data import (
    BookMultimodalDataset,
    TextVocabulary,
    build_label_mapping,
    compose_text,
    filter_records_with_images,
    load_book_records,
)
from bookgenre.inference import extract_embeddings, predict_dataset, write_jsonl
from bookgenre.model import MultimodalGenreModel
from bookgenre.training import TrainingConfig, train_model


In [12]:
# Main configuration cell
TRAIN_CSV = PROJECT_ROOT / "data" / "uchidalab_task1" / "book30-listing-train.csv"
VALID_CSV = PROJECT_ROOT / "data" / "uchidalab_task1" / "book30-listing-test.csv"
IMAGE_ROOT = PROJECT_ROOT.parent / "title30cat" / "224x224"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "notebook_runs" / "bookcover30_longrun"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32
EPOCHS = 20
NUM_WORKERS = 0  # On Windows notebooks, 0 is the safest starting point.
USE_AMP = DEVICE == "cuda"
INCLUDE_AUTHOR = True
BACKBONE = "resnet18"
PRETRAINED_BACKBONE = True
MAX_LENGTH = 32
MAX_VOCAB = 20000
TEXT_HIDDEN_DIM = 128
PROJECTION_DIM = 256
DROPOUT = 0.3
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

config_snapshot = {
    "train_csv": str(TRAIN_CSV),
    "valid_csv": str(VALID_CSV),
    "image_root": str(IMAGE_ROOT),
    "output_dir": str(OUTPUT_DIR),
    "device": DEVICE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "num_workers": NUM_WORKERS,
    "use_amp": USE_AMP,
    "include_author": INCLUDE_AUTHOR,
    "backbone": BACKBONE,
    "pretrained_backbone": PRETRAINED_BACKBONE,
}
print(json.dumps(config_snapshot, indent=2))


{
  "train_csv": "C:\\Users\\reddy\\OneDrive\\Documents\\Visual Code\\Book Genre Classification\\Multimodal-Book-Genre-Classification-Clustering\\data\\uchidalab_task1\\book30-listing-train.csv",
  "valid_csv": "C:\\Users\\reddy\\OneDrive\\Documents\\Visual Code\\Book Genre Classification\\Multimodal-Book-Genre-Classification-Clustering\\data\\uchidalab_task1\\book30-listing-test.csv",
  "image_root": "C:\\Users\\reddy\\OneDrive\\Documents\\Visual Code\\Book Genre Classification\\title30cat\\224x224",
  "output_dir": "C:\\Users\\reddy\\OneDrive\\Documents\\Visual Code\\Book Genre Classification\\Multimodal-Book-Genre-Classification-Clustering\\outputs\\notebook_runs\\bookcover30_longrun",
  "device": "cuda",
  "batch_size": 32,
  "epochs": 20,
  "num_workers": 0,
  "use_amp": true,
  "include_author": true,
  "backbone": "resnet18",
  "pretrained_backbone": true
}


In [13]:
# Dataset and loader setup
train_records = load_book_records(TRAIN_CSV)
valid_records = load_book_records(VALID_CSV)

train_records, missing_train = filter_records_with_images(train_records, IMAGE_ROOT)
valid_records, missing_valid = filter_records_with_images(valid_records, IMAGE_ROOT)

label_to_index = build_label_mapping(train_records + valid_records)
texts = [compose_text(record, include_author=INCLUDE_AUTHOR) for record in train_records]
vocabulary = TextVocabulary.build(texts, max_tokens=MAX_VOCAB)

train_dataset = BookMultimodalDataset(
    records=train_records,
    image_root=IMAGE_ROOT,
    label_to_index=label_to_index,
    vocabulary=vocabulary,
    max_length=MAX_LENGTH,
    include_author=INCLUDE_AUTHOR,
)
valid_dataset = BookMultimodalDataset(
    records=valid_records,
    image_root=IMAGE_ROOT,
    label_to_index=label_to_index,
    vocabulary=vocabulary,
    max_length=MAX_LENGTH,
    include_author=INCLUDE_AUTHOR,
)

pin_memory = DEVICE == "cuda"
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)

print("Train examples:", len(train_dataset))
print("Valid examples:", len(valid_dataset))
print("Missing train images:", len(missing_train))
print("Missing valid images:", len(missing_valid))
print("Vocabulary size:", len(vocabulary))
print("Num labels:", len(label_to_index))


Train examples: 51300
Valid examples: 5700
Missing train images: 0
Missing valid images: 0
Vocabulary size: 20000
Num labels: 30


In [14]:
# Training cell
model = MultimodalGenreModel(
    num_classes=len(label_to_index),
    vocab_size=len(vocabulary),
    backbone_name=BACKBONE,
    pretrained_backbone=PRETRAINED_BACKBONE,
    text_hidden_dim=TEXT_HIDDEN_DIM,
    projection_dim=PROJECTION_DIM,
    dropout=DROPOUT,
    use_text=True,
)

config = TrainingConfig(
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    device=DEVICE,
    use_amp=USE_AMP,
)

summary = train_model(model, train_loader, valid_loader, config=config, output_dir=OUTPUT_DIR)

metadata = {
    "num_classes": len(label_to_index),
    "vocab_size": len(vocabulary),
    "backbone_name": BACKBONE,
    "text_hidden_dim": TEXT_HIDDEN_DIM,
    "projection_dim": PROJECTION_DIM,
    "dropout": DROPOUT,
    "use_text": True,
    "max_length": MAX_LENGTH,
    "include_author": INCLUDE_AUTHOR,
    "missing_train_images": len(missing_train),
    "missing_valid_images": len(missing_valid),
    "train_examples": len(train_dataset),
    "validation_examples": len(valid_dataset),
    "device": DEVICE,
    "amp": USE_AMP,
    "num_workers": NUM_WORKERS,
}

save_checkpoint_bundle(OUTPUT_DIR, model, label_to_index, vocabulary, metadata)
(OUTPUT_DIR / "data_summary.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print(json.dumps(summary, indent=2))


C:\Users\reddy\OneDrive\Documents\Visual Code\Book Genre Classification\Multimodal-Book-Genre-Classification-Clustering\src\bookgenre\training.py:85: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this e

Resuming training from epoch 2/20


Epoch 2/20 [train]:   0%|                                                                 | 0/1604 [00:00<?, ?…

Epoch 2/20 [val]:   0%|                                                                    | 0/179 [00:00<?, ?…

Epoch 2/20 train_loss=2.3425 train_acc=0.3329 val_loss=2.3338 val_acc=0.3302 best_epoch=2 lr=0.000100 elapsed=503.8s speed=113.1 samples/s


...

Epoch 5/20 [train]:   0%|                                                                 | 0/1604 [00:00<?, ?…

Epoch 5/20 [val]:   0%|                                                                    | 0/179 [00:00<?, ?…

Epoch 5/20 train_loss=1.3179 train_acc=0.6013 val_loss=2.4815 val_acc=0.3560 best_epoch=3 lr=0.000050 elapsed=1823.3s speed=31.3 samples/s
Early stopping triggered after epoch 5. Best epoch was 3 with val_loss=2.3075.


C:\Users\reddy\OneDrive\Documents\Visual Code\Book Genre Classification\Multimodal-Book-Genre-Classification-Clustering\src\bookgenre\training.py:245: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this 

{
  "history": {
    "train_loss": [
      2.7751493859244603,
      2.342536311010171,
      2.0483828781455116,
      1.704230866051092,
      1.3178993101287306
    ],
    "train_accuracy": [
      0.22212475633528264,
      0.33286549707602336,
      0.40707602339181287,
      0.49692007797270954,
      0.6013450292397661
    ],
    "validation_loss": [
      2.4758114108704685,
      2.3337788163569937,
      2.307462393777412,
      2.394613970037092,
      2.4815472552650855
    ],
    "validation_accuracy": [
      0.2963157894736842,
      0.33017543859649123,
      0.34719298245614033,
      0.35,
      0.35596491228070176
    ],
    "learning_rate": [
      0.0001,
      0.0001,
      0.0001,
      0.0001,
      5e-05
    ]
  },
  "best_validation_loss": 2.307462393777412,
  "best_validation_accuracy": 0.34719298245614033,
  "best_epoch": 3,
  "final_validation_accuracy": 0.35596491228070176,
  "completed_epochs": 5,
  "requested_epochs": 20,
  "resumed": true,
  "resumed_fr

In [15]:
# Prediction export cell
prediction_output = OUTPUT_DIR / "test_predictions.jsonl"
predictions = predict_dataset(
    model=model,
    loader=valid_loader,
    label_to_index=label_to_index,
    device=DEVICE,
    top_k=3,
    review_threshold=0.55,
)
write_jsonl(predictions, prediction_output)
print("Predictions written to:", prediction_output)
print("Prediction rows:", len(predictions))
predictions[:2]


Predictions written to: C:\Users\reddy\OneDrive\Documents\Visual Code\Book Genre Classification\Multimodal-Book-Genre-Classification-Clustering\outputs\notebook_runs\bookcover30_longrun\test_predictions.jsonl
Prediction rows: 5700


[{'asin': '044310073X',
  'filename': '044310073X.jpg',
  'image_url': 'http://ecx.images-amazon.com/images/I/41kSLEoswsL.jpg',
  'title': 'Oral and Maxillofacial Surgery: An Objective-Based Textbook, 2e',
  'author': '',
  'category_id': '16',
  'category_name': 'Medical Books',
  'predicted_genre': 'Medical Books',
  'confidence': 0.682055,
  'needs_review': False,
  'image_weight': 0.528744,
  'top_k': [{'genre': 'Medical Books', 'score': 0.682055},
   {'genre': 'Law', 'score': 0.106314},
   {'genre': 'Health, Fitness & Dieting', 'score': 0.034742}]},
 {'asin': '1438005687',
  'filename': '1438005687.jpg',
  'image_url': 'http://ecx.images-amazon.com/images/I/510l0qhi01L.jpg',
  'title': "Barron's GRE, 21st Edition",
  'author': 'Sharon Weiner Green M.A.',
  'category_id': '28',
  'category_name': 'Test Preparation',
  'predicted_genre': 'Test Preparation',
  'confidence': 0.998637,
  'needs_review': False,
  'image_weight': 0.575613,
  'top_k': [{'genre': 'Test Preparation', 'score

In [16]:
# Optional clustering cell
embedding_rows = extract_embeddings(model, loader=valid_loader, device=DEVICE)
embeddings = torch.tensor([row["embedding"] for row in embedding_rows], dtype=torch.float32)
assignments, centroids = cluster_embeddings(embeddings, num_clusters=30)
prototypes = find_cluster_prototypes(embeddings, assignments, centroids)
silhouette = sample_silhouette_score(embeddings, assignments, sample_size=512)

print("Approx silhouette:", round(float(silhouette), 6))
print("Prototype count:", len(prototypes))


Approx silhouette: 0.080131
Prototype count: 30
